# <font color='cornflowerblue'>Child Mind Insitute Problematic Internet Use Approach UM - Data Team Club

In the third UM Data Team Club project, we'll predict the level of problematic internet usage exhibited by children and adolescents. The goal of this project is to develop a predictive model that analyzes children's physical activity and fitness data to identify early signs of problematic internet use.

<nav>
<a href=”https://uy.linkedin.com/company/data-team-club?trk=public_post_feed-actor-image">LinkedIn</a> |
<a href=”https://github.com/datateamclub”>Github</a>
</nav>

## Contents Table
 - [Imports](#1)
 - [Load Data](#3)
 - [Exploratory Data Analysis](#4)
     - [Severity Impairment Index Analysis](#4.1)
 - [Data Preprocessing](#5)
 - [Model](#6)

### Imports <a id='1'></a>

In [180]:
import numpy as np
import missingno as msno 
import math
import os
import pandas as pd
pd.set_option('display.max_columns', None)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import warnings
warnings.filterwarnings('ignore')

In [181]:
import numpy as np
import polars as pl
import pandas as pd
from sklearn.base import clone
from copy import deepcopy
import optuna
from scipy.optimize import minimize
import os
import matplotlib.pyplot as plt
import seaborn as sns

import re
from colorama import Fore, Style

from tqdm import tqdm
from IPython.display import clear_output
from concurrent.futures import ThreadPoolExecutor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

import lightgbm as lgb
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import *
from sklearn.metrics import *

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

### Load Data <a id='3'></a>

In [182]:
df_train = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/train.csv')
df_train.head()

,id,Basic_Demos-Enroll_Season,Basic_Demos-Age,Basic_Demos-Sex,CGAS-Season,CGAS-CGAS_Score,Physical-Season,Physical-BMI,Physical-Height,Physical-Weight,Physical-Waist_Circumference,Physical-Diastolic_BP,Physical-HeartRate,Physical-Systolic_BP,Fitness_Endurance-Season,Fitness_Endurance-Max_Stage,Fitness_Endurance-Time_Mins,Fitness_Endurance-Time_Sec,FGC-Season,FGC-FGC_CU,FGC-FGC_CU_Zone,FGC-FGC_GSND,FGC-FGC_GSND_Zone,FGC-FGC_GSD,FGC-FGC_GSD_Zone,FGC-FGC_PU,FGC-FGC_PU_Zone,FGC-FGC_SRL,FGC-FGC_SRL_Zone,FGC-FGC_SRR,FGC-FGC_SRR_Zone,FGC-FGC_TL,FGC-FGC_TL_Zone,BIA-Season,BIA-BIA_Activity_Level_num,BIA-BIA_BMC,BIA-BIA_BMI,BIA-BIA_BMR,BIA-BIA_DEE,BIA-BIA_ECW,BIA-BIA_FFM,BIA-BIA_FFMI,BIA-BIA_FMI,BIA-BIA_Fat,BIA-BIA_Frame_num,BIA-BIA_ICW,BIA-BIA_LDM,BIA-BIA_LST,BIA-BIA_SMM,BIA-BIA_TBW,PAQ_A-Season,PAQ_A-PAQ_A_Total,PAQ_C-Season,PAQ_C-PAQ_C_Total,PCIAT-Season,PCIAT-PCIAT_01,PCIAT-PCIAT_02,PCIAT-PCIAT_03,PCIAT-PCIAT_04,PCIAT-PCIAT_05,PCIAT-PCIAT_06,PCIAT-PCIAT_07,PCIAT-PCIAT_08,PCIAT-PCIAT_09,PCIAT-PCIAT_10,PCIAT-PCIAT_11,PCIAT-PCIAT_12,PCIAT-PCIAT_13,PCIAT-PCIAT_14,PCIAT-PCIAT_15,PCIAT-PCIAT_16,PCIAT-PCIAT_17,PCIAT-PCIAT_18,PCIAT-PCIAT_19,PCIAT-PCIAT_20,PCIAT-PCIAT_Total,SDS-Season,SDS-SDS_Total_Raw,SDS-SDS_Total_T,PreInt_EduHx-Season,PreInt_EduHx-computerinternet_hoursday,sii
0,00008ff9,Fall,5,0,Winter,51.0,Fall,16.877316,46.0,50.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fall,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,7.0,0.0,6.0,0.0,6.0,1.0,Fall,2.0,2.66855,16.8792,932.498,1492.00,8.25598,41.5862,13.8177,3.06143,9.21377,1.0,24.4349,8.89536,38.9177,19.5413,32.6909,NaN,NaN,NaN,NaN,Fall,5.0,4.0,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,4.0,0.0,4.0,4.0,4.0,4.0,4.0,4.0,2.0,4.0,55.0,NaN,NaN,NaN,Fall,3.0,2.0
1,000fd460,Summer,9,0,NaN,NaN,Fall,14.035590,48.0,46.0,22.0,75.0,70.0,122.0,NaN,NaN,NaN,NaN,Fall,3.0,0.0,NaN,NaN,NaN,NaN,5.0,0.0,11.0,1.0,11.0,1.0,3.0,0.0,Winter,2.0,2.57949,14.0371,936.656,1498.65,6.01993,42.0291,12.8254,1.21172,3.97085,1.0,21.0352,14.97400,39.4497,15.4107,27.0552,NaN,NaN,Fall,2.340,Fall,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Fall,46.0,64.0,Summer,0.0,0.0
2,00105258,Summer,10,1,Fall,71.0,Fall,16.648696,56.5,75.6,NaN,65.0,94.0,117.0,Fall,5.0,7.0,33.0,Fall,20.0,1.0,10.2,1.0,14.7,2.0,7.0,1.0,10.0,1.0,10.0,1.0,5.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Summer,2.170,Fall,5.0,2.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,28.0,Fall,38.0,54.0,Summer,2.0,0.0
3,00115b9f,Winter,9,0,Fall,71.0,Summer,18.292347,56.0,81.6,NaN,60.0,97.0,117.0,Summer,6.0,9.0,37.0,Summer,18.0,1.0,NaN,NaN,NaN,NaN,5.0,0.0,7.0,0.0,7.0,0.0,7.0,1.0,Summer,3.0,3.84191,18.2943,1131.430,1923.44,15.59250,62.7757,14.0740,4.22033,18.82430,2.0,30.4041,16.77900,58.9338,26.4798,45.9966,NaN,NaN,Winter,2.451,Summer,4.0,2.0,4.0,0.0,5.0,1.0,0.0,3.0,2.0,2.0,3.0,0.0,3.0,0.0,0.0,3.0,4.0,3.0,4.0,1.0,44.0,Summer,31.0,45.0,Winter,0.0,1.0
4,0016bb22,Spring,18,1,Summer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Summer,1.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [183]:
df_test = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/test.csv')

In [ ]:
def process_file(filename, dirname):
    df = pd.read_parquet(os.path.join(dirname, filename, 'part-0.parquet'))
    df.drop('step', axis=1, inplace=True)
    return df.describe().values.reshape(-1), filename.split('=')[1]

def load_time_series(dirname) -> pd.DataFrame:
    ids = os.listdir(dirname)
    
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(lambda fname: process_file(fname, dirname), ids), total=len(ids)))
    
    stats, indexes = zip(*results)
    
    df = pd.DataFrame(stats, columns=[f"Stat_{i}" for i in range(len(stats[0]))])
    df['id'] = indexes
    
    return df


train_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_train.parquet")
test_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_test.parquet")
time_series_cols = train_ts.columns.tolist()
time_series_cols.remove("id")

 91%|█████████ | 907/996 [01:09<00:05, 17.35it/s]

## Time series treatment

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim*3),
            nn.LeakyReLU(0.2),
            nn.Linear(encoding_dim*3, encoding_dim*2),
            nn.LeakyReLU(0.2),
            nn.Linear(encoding_dim*2, encoding_dim),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim*2),
            nn.LeakyReLU(0.2),
            nn.Linear(input_dim*2, input_dim*3),
            nn.LeakyReLU(0.2),
            nn.Linear(input_dim*3, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
def perform_autoencoder(df, encoding_dim=50, epochs=50, batch_size=32):
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df)

    data_tensor = torch.FloatTensor(df_scaled)

    input_dim = data_tensor.shape[1]
    autoencoder = AutoEncoder(input_dim, encoding_dim)

    criterion = F.smooth_l1_loss
    optimizer = optim.Adam(autoencoder.parameters())

    for epoch in range(epochs):
        for i in range(0, len(data_tensor), batch_size):
            batch = data_tensor[i : i + batch_size]
            optimizer.zero_grad()
            reconstructed = autoencoder(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}]')

    with torch.no_grad():
        encoded_data = autoencoder.encoder(data_tensor).numpy()

    df_encoded = pd.DataFrame(encoded_data, columns=[f'Enc_{i + 1}' for i in range(encoded_data.shape[1])])

    return df_encoded

In [ ]:
train_ts_temp = train_ts.drop('id', axis=1)
test_ts_temp = test_ts.drop('id', axis=1)

train_ts_encoded = perform_autoencoder(train_ts_temp, encoding_dim=60, epochs=100, batch_size=32)
test_ts_encoded = perform_autoencoder(test_ts_temp, encoding_dim=60, epochs=100, batch_size=32)

time_series_cols = train_ts_encoded.columns.tolist()

train_ts_encoded["id"] = train_ts["id"]
test_ts_encoded["id"] = test_ts["id"]
train_ts = train_ts_encoded
test_ts = test_ts_encoded

In [ ]:
df_train = pd.merge(df_train, train_ts, how="left", on='id')
df_test = pd.merge(df_test, test_ts, how="left", on='id')

### **Target Variable (SII)**

- **0: None**  
  *PCIAT-PCIAT_Total from 0 to 30*
- **1: Mild**  
  *PCIAT-PCIAT_Total from 31 to 49*
- **2: Moderate**  
  *PCIAT-PCIAT_Total from 50 to 79*
- **3: Severe**  
  *PCIAT-PCIAT_Total 80 and more*

### Exploratory Data Analysis <a id='4'></a>

In [ ]:
df_train.shape

In [ ]:
(df_train.isna().mean() * 100).sort_values(ascending=False).head(15)

In [ ]:
df_train.select_dtypes(include=['number']).corr()['sii'][25:].sort_values(ascending=False)

In [ ]:
msno.dendrogram(df_train)

<div class="alert alert-block alert-info">
<b></b> We will review the columns available in the test dataframe to ensure predictions can be made without relying on variables that are missing.</div>



In [ ]:
columns_useful_to_predict = [col for col in list(df_train.columns) if col in list(df_test.columns)]

print(f'The variables available for prediction are: \n {columns_useful_to_predict}, being in total {len(columns_useful_to_predict)}.')

### **Severity Impairment Index Analysis** <a id="4.1"></a>

In [ ]:
labels_map = {0: 'None', 1: 'Mild', 2: 'Moderate', 3: 'Severe', -1: 'Missing'}
label_counts = df_train['sii'].map(lambda x: labels_map.get(x, 'Missing')).value_counts()

def func(pct, allvals):
    absolute = int(pct / 100. * sum(allvals))
    return f"{absolute} ({pct:.1f}%)"

plt.figure(figsize=(10, 5))
wedges, texts, autotexts = plt.pie(
    label_counts, 
    autopct=lambda pct: func(pct, label_counts), 
    labels=[None] * len(label_counts), 
    colors=plt.cm.Pastel1.colors)

plt.legend(
    wedges, 
    label_counts.index, 
    title="Labels", 
    loc="center left", 
    bbox_to_anchor=(1, 0.5)
)
plt.title("Distribution of Severity Impairment Index")
display()

In [ ]:
plt.figure(figsize=(10, 5))
n, bins, patches = plt.hist(
    df_train['PCIAT-PCIAT_Total'], 
    bins=10,
    color = plt.cm.Pastel1.colors[1])

plt.title("Distribution of PCIAT-PCIAT_Total")
display()

#### Is there an association between gender and the Severity Impairment Index?

In [ ]:
colors = [plt.cm.Pastel1.colors[1], plt.cm.Pastel1.colors[0]]
gender_labels = {0: 'Male', 1: 'Female'}
gender_counts = df_train['Basic_Demos-Sex'].map(lambda x: gender_labels.get(x, 'Unknown')).value_counts()

plt.figure(figsize=(10, 5))
wedges, texts, autotexts = plt.pie(
    gender_counts, 
    autopct=lambda pct: func(pct, gender_counts), 
    labels=[None] * len(gender_counts), 
    colors=colors
)

plt.legend(
    wedges, 
    gender_counts.index, 
    title="Gender", 
    loc="center left", 
    bbox_to_anchor=(1, 0.5)
)

plt.title("Gender Distribution")
plt.show()

In [ ]:
def plot_pie(data, gender_label, ax):
    label_counts = data['sii'].map(lambda x: labels_map.get(x, 'Missing')).value_counts()
    
    color_count = len(label_counts)
    pie_colors = colors[:color_count]

    wedges, texts, autotexts = ax.pie(
        label_counts, 
        autopct=lambda pct: func(pct, label_counts), 
        labels=[None] * len(label_counts), 
        colors=plt.cm.Pastel1.colors, 
        textprops={'fontsize': 18}
    )
    
    for autotext in autotexts:
        autotext.set_fontsize(18)
    
    ax.set_title(f"Distribution of Severity Impairment Index {gender_label}", fontsize=18)
    
    return wedges

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

all_wedges = []
gender_labels_list = []

for i, (gender_value, gender_label) in enumerate(gender_labels.items()):
    gender_data = df_train[df_train['Basic_Demos-Sex'] == gender_value]
    wedges = plot_pie(gender_data, gender_label, axes[i])
    all_wedges.extend(wedges)
    gender_labels_list.append(gender_label)

fig.legend(
    all_wedges, 
    label_counts.index, 
    title="Labels", 
    loc="center left", 
    bbox_to_anchor=(1, 0.5),
    fontsize=18,
    title_fontsize=18
)

plt.tight_layout()
display()

In [ ]:
categories = list(labels_map.values())

male_values = (
    df_train[df_train['Basic_Demos-Sex'] == 0]['sii']
    .map(lambda x: labels_map.get(x, 'Missing'))
    .value_counts()
    .reindex(categories, fill_value=0)
)

female_values = (
    df_train[df_train['Basic_Demos-Sex'] == 1]['sii']
    .map(lambda x: labels_map.get(x, 'Missing'))
    .value_counts()
    .reindex(categories, fill_value=0)
)

x = np.arange(len(categories))
width = 0.4

fig, ax = plt.subplots(figsize=(12, 6))

bars_male = ax.bar(x - width / 2, male_values, width, label='Male', color=plt.cm.Pastel1.colors[1])
bars_female = ax.bar(x + width / 2, female_values, width, label='Female', color=plt.cm.Pastel1.colors[0])

ax.set_title('Comparison of Severity Impairment Index by Gender')
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=0, ha='right')
ax.legend()

for bars in (bars_male, bars_female):
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2, 
                height, 
                str(height), 
                ha='center', 
                va='bottom'
            )

plt.tight_layout()
display()

<div class="alert alert-block alert-info">
<b></b> Overall, the data indicates that males have a slightly higher likelihood of being categorized under the "Mild," "Moderate," and "Severe" levels of impairment compared to females, while females are more likely to fall into the "None" category. These findings suggest a potential relationship between gender and the Severity Impairment Index (SII). Further analysis will be conducted, incorporating other variables, to gain a deeper understanding of the factors influencing the Severity Impairment Index.</div>

#### How does the distribution of the SII vary across different age groups?

To conclude the analysis of the demographic variables, it is important to examine the age of the children and adolescents to determine if there is a relationship with the Severity Impairment Index. Understanding how SII varies across age groups can provide insights into potential trends or patterns related to developmental stages.

In [ ]:
df_train['Basic_Demos-Age'].describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_train['Basic_Demos-Age'], color=plt.cm.Pastel1.colors[1], bins=range(int(df_train['Basic_Demos-Age'].min()), 
               int(df_train['Basic_Demos-Age'].max()) + 1))

plt.title("Age Distribution")
plt.xticks(np.arange(int(df_train['Basic_Demos-Age'].min()), int(df_train['Basic_Demos-Age'].max()) + 1, step=1))
display()

## Feature engeneering]

In [ ]:
def remove_outliers(df):
    
    df = df.drop(df[df['Physical-BMI'] <= 0].index)
    df = df.drop(df[df['Physical-Diastolic_BP'] <= 0].index)
    df = df.drop(df[df['Physical-Systolic_BP'] <= 0].index)
    df = df.drop(df[df['Physical-Diastolic_BP'] > 160].index)

    children = df[df['Basic_Demos-Age'] <= 12]
    df = df.drop(children[children['FGC-FGC_CU'] > 80].index)
    df = df.drop(children[children['FGC-FGC_GSND'] > 80].index)

    df = df.drop(df[df['BIA-BIA_BMI'] <= 0].index)
    df = df.drop(df[df['BIA-BIA_BMC'] > 1000].index)
    df = df.drop(df[df['BIA-BIA_BMR'] > 40000].index)
    df = df.drop(df[df['BIA-BIA_DEE'] > 60000].index)
    df = df.drop(df[df['BIA-BIA_ECW'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_FFM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_ICW'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_LDM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_LST'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_SMM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_TBW'] > 2000].index)
    
    return df

def feature_engineering(df):
    season_cols = [col for col in df.columns if 'Season' in col]
    df = df.drop(season_cols, axis=1) 
    df['BMI_Age'] = df['Physical-BMI'] * df['Basic_Demos-Age']
    df['Internet_Hours_Age'] = df['PreInt_EduHx-computerinternet_hoursday'] * df['Basic_Demos-Age']
    df['BMI_Internet_Hours'] = df['Physical-BMI'] * df['PreInt_EduHx-computerinternet_hoursday']
    df['BFP_BMI'] = df['BIA-BIA_Fat'] / df['BIA-BIA_BMI']
    df['FFMI_BFP'] = df['BIA-BIA_FFMI'] / df['BIA-BIA_Fat']
    df['FMI_BFP'] = df['BIA-BIA_FMI'] / df['BIA-BIA_Fat']
    df['LST_TBW'] = df['BIA-BIA_LST'] / df['BIA-BIA_TBW']
    df['BFP_BMR'] = df['BIA-BIA_Fat'] * df['BIA-BIA_BMR']
    df['BFP_DEE'] = df['BIA-BIA_Fat'] * df['BIA-BIA_DEE']
    df['BMR_Weight'] = df['BIA-BIA_BMR'] / df['Physical-Weight']
    df['DEE_Weight'] = df['BIA-BIA_DEE'] / df['Physical-Weight']
    df['SMM_Height'] = df['BIA-BIA_SMM'] / df['Physical-Height']
    df['Muscle_to_Fat'] = df['BIA-BIA_SMM'] / df['BIA-BIA_FMI']
    df['Hydration_Status'] = df['BIA-BIA_TBW'] / df['Physical-Weight']
    df['ICW_TBW'] = df['BIA-BIA_ICW'] / df['BIA-BIA_TBW']
    
    return df

In [ ]:
df_train_temp = remove_outliers(df_train)
df_test_temp = remove_outliers(df_test)
df_train = feature_engineering(df_train_temp)
df_test = feature_engineering(df_test_temp)

## Modelado basico

In [ ]:
www

In [ ]:
columns_to_encode = [col for col in df_train.columns if "Season" in col]
# Realizar one-hot encoding
df_encoded_train = pd.get_dummies(df_train, columns=columns_to_encode)

columns_to_encode = [col for col in df_test.columns if "Season" in col]
# Realizar one-hot encoding
df_encoded_test = pd.get_dummies(df_test, columns=columns_to_encode)

In [ ]:
columns_useful_to_predict = [col for col in list(df_encoded_train.columns) if col in list(df_encoded_test.columns)]
print(df_encoded_train.shape)
df_encoded_train = df_encoded_train[columns_useful_to_predict + ['sii', 'PCIAT-PCIAT_Total']]
print(df_encoded_train.shape)

In [ ]:
df_encoded_train = df_encoded_train.drop(columns=['id'])

Knn evaluando diferentes k neighbours

In [ ]:
from sklearn.impute import KNNImputer

df_encoded = df_encoded_train.dropna(subset=['sii'])

X = df_encoded.drop(columns = ['sii', 'PCIAT-PCIAT_Total'])
y = df_encoded['sii']  

imputer = KNNImputer(n_neighbors=2)  # numero 2 optimo -pruebas manuales- -perdon-
X_imputed = imputer.fit_transform(X)

In [ ]:
X

First model accuracy score of 0.60, overall competition performance 0.20.

In [ ]:
print(df_encoded.corr()['sii'].sort_values(ascending = False))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier

# Suponiendo que X_imputed y y ya están definidos
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.3, random_state=42)

# Diccionario de modelos
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

# DataFrame para almacenar resultados
results = []

# Evaluar cada modelo
for name, model in models.items():
    model.fit(X_train, y_train)  # Entrenar el modelo
    y_pred = model.predict(X_test)  # Predicciones
    
    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Guardar resultados en una lista
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

# Convertir resultados a un DataFrame
comparison_table = pd.DataFrame(results)

# Mostrar tabla ordenada por Accuracy
comparison_table = comparison_table.sort_values(by="Accuracy", ascending=False)
print(comparison_table)

### Modelo con clases train-test balanceadas 
(args- stratify = y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.3, random_state=42, stratify = y)

# Diccionario de modelos
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

# DataFrame para almacenar resultados
results = []

# Evaluar cada modelo
for name, model in models.items():
    model.fit(X_train, y_train)  # Entrenar el modelo
    y_pred = model.predict(X_test)  # Predicciones
    
    # Calcular métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Guardar resultados en una lista
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

# Convertir resultados a un DataFrame
comparison_table = pd.DataFrame(results)

# Mostrar tabla ordenada por Accuracy
comparison_table = comparison_table.sort_values(by="Accuracy", ascending=False)
print(comparison_table)

#### Train Submission Model

In [ ]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train) 

## Final submission ensambling

In [ ]:
# predicciones con el primer modelo

df_submission = pd.DataFrame()

df_submission['id'] = df_test['id']
print(df_submission.shape)

# label encoding
columns_to_encode = [col for col in df_test.columns if "Season" in col]
df_encoded = pd.get_dummies(df_test, columns=columns_to_encode)

X = df_encoded.drop(columns = ['id'])

# manejo missing vals
imputer = KNNImputer(n_neighbors=2)  # Puedes ajustar el número de vecinos
X_imputed_final = imputer.fit_transform(X)

y_pred = model.predict(X_imputed_final)

df_submission['sii'] = y_pred

In [ ]:
df_submission['sii'] = df_submission['sii'].astype('int64')
df_submission.head()

In [ ]:
df_submission.to_csv('submission.csv', index = False)

In [ ]:
df_submission.sii.value_counts()

(Vs una regresion, no dio buenos resultados.

## Opciones para seguir probando:
1. usar los datos de los monitores continuos (parquets)
2. feature engeneering
3. elegir otro modelo
4. fine tunning de parametros
5. trabajar el class imbalance, revisar si en el trainn y test no quedan sesgados y faltan  clases -representatividad-